# Создание моделей для предсказания свойств углепластика, полученного по автоклавной технологии технологии

In [1]:
import pandas as pd
import numpy as np


# инструменты для построения модели:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression # инструмент для создания и обучения модели
from sklearn.ensemble import RandomForestRegressor # инструмент для создания и обучения модели
from sklearn import metrics # инструменты для оценки точности модели
from xgboost import XGBRegressor

import matplotlib as plt

import pickle

from sklearn.utils import shuffle

RANDOM_SEED = 42

In [2]:
df = pd.read_csv('data/dataset_prepaired.csv')
df.head()

,Linera density,Density yarn,Strength Gpa,Module Gpa,lengthening,Mass size,breaking the loop,Surface density of the fabric,Prepreg surface density,Resin content,viscosity,Gelation time,Resin Tg,technology,Thickness of the monolayer,density,Strength_plastik,Module_plastik,LSS,Plastik_Tg
0,188.0,1.758,4.59,253.0,1.814229,1.0,24.3,196.0,319.4,38.63,23.672,19.0,150.2,0,0.209,1.536,879.0,70.1,77.2,164.0
1,189.0,1.758,4.48,260.0,1.723077,1.1,24.3,196.0,319.4,38.63,23.672,19.0,150.2,0,0.209,1.536,879.0,70.1,77.2,164.0
2,188.0,1.759,4.28,257.0,1.665370,1.0,24.3,196.0,319.4,38.63,23.672,19.0,150.2,0,0.209,1.536,879.0,70.1,77.2,164.0
3,187.0,1.758,4.77,256.0,1.863281,1.0,24.3,196.0,319.4,38.63,23.672,19.0,150.2,0,0.209,1.536,879.0,70.1,77.2,164.0
4,190.0,1.757,4.56,255.0,1.788235,0.9,24.3,196.0,319.4,38.63,23.672,19.0,150.2,0,0.209,1.536,879.0,70.1,77.2,164.0


In [3]:

df = df[df['technology'] == 1]
df = df.drop('technology', axis=1)

In [4]:
df = df.rename(columns={'Thickness of the monolayer' : 'Thickness_monolayer',
                        'Module_plastik ' : 'Module_plastik'})

In [5]:
df = shuffle(df)

In [6]:
x1 = np.array([[185.62, 1.769, 4.56, 261, 1.75, 1.07, 26.8, 200, 325.7, 38.57, 32.35, 12, 153]])
x2 = np.array([[185.62, 1.769, 4.56, 261, 1.75, 1.07, 26.8, 200, 330.5, 39.43, 30.85, 10.82, 150.36]])

### Модель для предсказания толщины монослоя Thickness_monolayer

In [7]:
train_data_thickness = df.drop(['density', 'Strength_plastik', 'Module_plastik',
       'LSS', 'Plastik_Tg'], axis=1)

X = np.array(train_data_thickness.drop(['Thickness_monolayer'], axis=1))
y = np.array(train_data_thickness.Thickness_monolayer.values)

In [8]:
# воспользуемся специальной функцие train_test_split для разделения тестовых данных
# выделим 20% данных на валидацию (параметр test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

In [9]:
# НАСТРОЙКИ 
model_rf_thikness = RandomForestRegressor(
    n_estimators=800,
    max_features=5,
    min_samples_leaf=5,
    max_depth=10,
    verbose=1, 
    n_jobs=-1,
    random_state=RANDOM_SEED)

In [10]:
# обучаем модель на тестовом наборе данных
model_rf_thikness.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_rf_thikness.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 426 tasks      | elapsed:    0.3s
[Parallel(n_jobs=-1)]: Done 776 tasks      | elapsed:    0.4s
[Parallel(n_jobs=-1)]: Done 800 out of 800 | elapsed:    0.5s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 426 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 776 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 800 out of 800 | elapsed:    0.1s finished


In [11]:
def mean_absolute_percentage_error(y_tr, y_pr):
    """Получение средней абсолютной ошибки"""
    y_tr, y_pr = np.array(y_tr), np.array(y_pr)
    return np.mean(np.abs((y_tr - y_pr) / y_tr)) * 100

In [12]:
# сравниваем предсказанные значения (y_pred) с реальными (y_test), 
# метрика mean squared error, MSE показывает среднеквадратичное отклонение:

def mean_squared_error(y_tr, y_pr):
    """Получение средней абсолютной ошибки"""
    y_tr, y_pr = np.array(y_tr), np.array(y_pr)
    return np.mean(np.abs((y_tr - y_pr)**2)) 

In [13]:
print('model_rf_thikness MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('model_rf_thikness MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

model_rf_thikness MSE: 0.0
model_rf_thikness MAPE: 0.032


In [14]:
# save model
with open('autoclave_model_rf_thikness.pkl','wb') as f:
    pickle.dump(model_rf_thikness,f)

In [15]:
model_lr_thikness = LinearRegression()

# обучаем модель на тестовом наборе данных
model_lr_thikness.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_lr_thikness.predict(X_test)

print('model_lr_thiknessMSE:', round(mean_squared_error(y_test, y_pred), 3))

print('model_lr_thikness MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

model_lr_thiknessMSE: 0.0
model_lr_thikness MAPE: 1.528


In [16]:
# save model
with open('autoclave_model_lr_thikness.pkl','wb') as f:
    pickle.dump(model_lr_thikness,f)

In [17]:
print('Линейная регрессия. Толщина монослоя:', model_lr_thikness.predict(x2))
print('Случайный лес регрессия. Толщина монослоя:', model_rf_thikness.predict(x2))

Линейная регрессия. Толщина монослоя: [0.21915448]
Случайный лес регрессия. Толщина монослоя: [0.20946932]


[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 426 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 776 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 800 out of 800 | elapsed:    0.0s finished


### Модель для предсказания плотности углепластика density

In [18]:
train_data_density = df.drop(['Thickness_monolayer', 'Strength_plastik', 'Module_plastik',
                            'LSS', 'Plastik_Tg'], axis=1)

X = np.array(train_data_density.drop(['density'], axis=1))
y = np.array(train_data_density.density.values)

In [19]:
# воспользуемся специальной функцие train_test_split для разделения тестовых данных
# выделим 20% данных на валидацию (параметр test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

# НАСТРОЙКИ 
model_rf_density = RandomForestRegressor(
    n_estimators=800,
    max_features=4,
    min_samples_leaf=5,
    max_depth=15,
    verbose=1, 
    n_jobs=-1, 
    random_state=RANDOM_SEED)

# обучаем модель на тестовом наборе данных
model_rf_density.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_rf_density.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 426 tasks      | elapsed:    0.3s
[Parallel(n_jobs=-1)]: Done 776 tasks      | elapsed:    0.5s
[Parallel(n_jobs=-1)]: Done 800 out of 800 | elapsed:    0.5s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 426 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 776 tasks      | elapsed:    0.1s
[Parallel(n_jobs=12)]: Done 800 out of 800 | elapsed:    0.1s finished


In [20]:
print('model_rf_density MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('model_rf_density MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

model_rf_density MSE: 0.0
model_rf_density MAPE: 0.005


In [21]:
# save model
with open('autoclave_model_rf_density.pkl','wb') as f:
    pickle.dump(model_rf_density,f)

In [22]:
model_lr_density = LinearRegression()

# обучаем модель на тестовом наборе данных
model_lr_density.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_lr_density.predict(X_test)

print('model_lr_density MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('model_lr_density MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

model_lr_density MSE: 0.0
model_lr_density MAPE: 0.424


In [23]:
# save model
with open('autoclave_model_lr_density.pkl','wb') as f:
    pickle.dump(model_lr_density,f)

In [24]:
print('Линейная регрессия. Плотность:', model_lr_density.predict(x2))
print('Случайный лес регрессия. Плотность:', model_rf_density.predict(x2))

[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 426 tasks      | elapsed:    0.0s


Линейная регрессия. Плотность: [1.54702636]
Случайный лес регрессия. Плотность: [1.54297327]


[Parallel(n_jobs=12)]: Done 776 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 800 out of 800 | elapsed:    0.0s finished


### Модель для предсказания прочности углепластика Strength

In [58]:
train_data_strength = df.drop(['Thickness_monolayer', 'Module_plastik',
       'LSS', 'Plastik_Tg', 'density'], axis=1)

X = np.array(train_data_strength.drop(['Strength_plastik'], axis=1))
y = np.array(train_data_strength.Strength_plastik.values)

In [59]:
# воспользуемся специальной функцие train_test_split для разделения тестовых данных
# выделим 20% данных на валидацию (параметр test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

In [67]:
# НАСТРОЙКИ 
model_rf_strength = RandomForestRegressor(
    n_estimators=1000,
    max_features=4,
    min_samples_leaf=5,
    max_depth=20,
    verbose=1, 
    n_jobs=-1, 
    random_state=RANDOM_SEED)

# обучаем модель на тестовом наборе данных
model_rf_strength.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_rf_strength.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 426 tasks      | elapsed:    0.3s
[Parallel(n_jobs=-1)]: Done 776 tasks      | elapsed:    0.5s
[Parallel(n_jobs=-1)]: Done 1000 out of 1000 | elapsed:    0.6s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 426 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 776 tasks      | elapsed:    0.1s
[Parallel(n_jobs=12)]: Done 1000 out of 1000 | elapsed:    0.1s finished


In [68]:
print('model_rf_strength MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('model_rf_strength MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

model_rf_strength MSE: 16.203
model_rf_strength MAPE: 0.04


In [69]:
# save model
with open('autoclave_model_rf_strength.pkl','wb') as f:
    pickle.dump(model_rf_strength,f)

In [70]:
model_lr_strength = LinearRegression()

# обучаем модель на тестовом наборе данных
model_lr_strength.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_lr_strength.predict(X_test)

print('model_lr_strength MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('model_lr_strength MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

model_lr_strength MSE: 561.714
model_lr_strength MAPE: 1.988


In [31]:
# save model
with open('autoclave_model_lr_strength.pkl','wb') as f:
    pickle.dump(model_lr_strength,f)

In [32]:
model_xgb_strenght = XGBRegressor(lerning_rate = 0.01)

# обучаем модель на тестовом наборе данных
model_xgb_strenght.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_xgb_strenght.predict(X_test)

print('model_xgb_strenghtMSE:', round(mean_squared_error(y_test, y_pred), 3))

print('model_xgb_strenght MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

model_xgb_strenghtMSE: 14.701
model_xgb_strenght MAPE: 0.027


/home/alexandr/anaconda3/envs/ptn312/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [22:52:23] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "lerning_rate" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [33]:
# save model
with open('autoclave_model_xgb_strenght.pkl','wb') as f:
    pickle.dump(model_xgb_strenght,f)

In [71]:
print('Линейная регрессия. Прочность:', model_lr_strength.predict(x2))
print('Случайный лес регрессия. Прочность:', model_rf_strength.predict(x2))
print('XGB регрессия. Прочность:', model_xgb_strenght.predict(x2))

Линейная регрессия. Прочность: [965.32950562]
Случайный лес регрессия. Прочность: [963.94601062]
XGB регрессия. Прочность: [909.531]


[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 426 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 776 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 1000 out of 1000 | elapsed:    0.1s finished


### Модель для предсказания модуля углепластика Module

In [35]:
train_data_module = df.drop(['Thickness_monolayer', 'density', 'Strength_plastik',
       'LSS', 'Plastik_Tg'], axis=1)

X = np.array(train_data_module.drop(['Module_plastik'], axis=1))
y = np.array(train_data_module.Module_plastik .values)

In [36]:
# воспользуемся специальной функцие train_test_split для разделения тестовых данных
# выделим 20% данных на валидацию (параметр test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

# НАСТРОЙКИ 
model_rf_module = RandomForestRegressor(
    n_estimators=500, 
    max_features=4,
    min_samples_leaf=5,
    max_depth=9,
    verbose=1, 
    n_jobs=-1, 
    random_state=RANDOM_SEED)

# обучаем модель на тестовом наборе данных
model_rf_module.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_rf_module.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 426 tasks      | elapsed:    0.3s
[Parallel(n_jobs=-1)]: Done 500 out of 500 | elapsed:    0.3s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 426 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 500 out of 500 | elapsed:    0.0s finished


In [37]:
print('model_rf_module MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('model_rf_module MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

model_rf_module MSE: 0.047
model_rf_module MAPE: 0.034


In [38]:
# save model
with open('autoclave_model_rf_module.pkl','wb') as f:
    pickle.dump(model_rf_module,f)

In [39]:
model_lr_module = LinearRegression()

# обучаем модель на тестовом наборе данных
model_lr_module.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_lr_module.predict(X_test)

print('model_lr_module MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('model_lr_moduleMAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

model_lr_module MSE: 3.169
model_lr_moduleMAPE: 2.249


In [40]:
# save model
with open('autoclave_model_lr_module.pkl','wb') as f:
    pickle.dump(model_lr_module,f)

In [41]:
print('Линейная регрессия. Модуль упругости:', model_lr_module.predict(x2))
print('Случайный лес регрессия. Модуль упругости:', model_rf_module.predict(x2))

Линейная регрессия. Модуль упругости: [69.65773229]
Случайный лес регрессия. Модуль упругости: [67.42116844]


[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 426 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 500 out of 500 | elapsed:    0.0s finished


### Модель для предсказания межслоевой прочности углепластика LSS

In [42]:
train_data_lss = df.drop(['Thickness_monolayer', 'density', 'Strength_plastik', 'Module_plastik',
                             'Plastik_Tg'], axis=1)

X = np.array(train_data_lss.drop(['LSS'], axis=1))
y = np.array(train_data_lss.LSS.values)

In [43]:
# воспользуемся специальной функцие train_test_split для разделения тестовых данных
# выделим 20% данных на валидацию (параметр test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

# НАСТРОЙКИ 
model_rf_lss = RandomForestRegressor(
    n_estimators=500,
    max_features=4,
    min_samples_leaf=5,
    max_depth=9,
    verbose=1, 
    n_jobs=-1, 
    random_state=RANDOM_SEED)

# обучаем модель на тестовом наборе данных
model_rf_lss.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_rf_lss.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 426 tasks      | elapsed:    0.3s
[Parallel(n_jobs=-1)]: Done 500 out of 500 | elapsed:    0.3s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 426 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 500 out of 500 | elapsed:    0.0s finished


In [44]:
print('model_rf_lss MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('model_rf_lss MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

model_rf_lss MSE: 0.078
model_rf_lss MAPE: 0.051


In [45]:
# save model
with open('autoclave_model_rf_lss.pkl','wb') as f:
    pickle.dump(model_rf_lss,f)

In [46]:
model_lr_lss = LinearRegression()

# обучаем модель на тестовом наборе данных
model_lr_lss.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_lr_lss.predict(X_test)

print('model_lr_lss MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('model_lr_lssMAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

model_lr_lss MSE: 18.47
model_lr_lssMAPE: 4.344


In [47]:
# save model
with open('autoclave_model_lr_lss.pkl','wb') as f:
    pickle.dump(model_lr_lss,f)

In [48]:
model_xgb_lss = XGBRegressor(lerning_rate = 0.01)

# обучаем модель на тестовом наборе данных
model_xgb_lss.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_xgb_lss.predict(X_test)

print('model_xgb_lss MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('model_xgb_lss MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

/home/alexandr/anaconda3/envs/ptn312/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [22:52:25] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "lerning_rate" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


model_xgb_lss MSE: 0.2
model_xgb_lss MAPE: 0.039


In [49]:
# save model
with open('autoclave_model_xgb_lss.pkl','wb') as f:
    pickle.dump(model_xgb_lss,f)

In [50]:
print('Линейная регрессия. Прочность при сдвиге:', model_lr_lss.predict(x1))
print('Случайный лес регрессия. Прочность при сдвиге:', model_rf_lss.predict(x1))
print('XGB регрессия. Прочность при сдвиге:', model_xgb_lss.predict(x1))

Линейная регрессия. Прочность при сдвиге: [76.89668167]
Случайный лес регрессия. Прочность при сдвиге: [76.61830448]
XGB регрессия. Прочность при сдвиге: [72.940926]


[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 426 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 500 out of 500 | elapsed:    0.0s finished


# Модель для предсказания температуры стеклования углепластика Tg

In [51]:
train_data_Tg = df.drop(['Thickness_monolayer', 'density', 'Strength_plastik', 'Module_plastik', 
                         'LSS'], axis=1)

X = np.array(train_data_Tg.drop(['Plastik_Tg'], axis=1))
y = np.array(train_data_Tg.Plastik_Tg.values)

In [52]:
# воспользуемся специальной функцие train_test_split для разделения тестовых данных
# выделим 20% данных на валидацию (параметр test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

# НАСТРОЙКИ 
model_rf_Tg = RandomForestRegressor(
    n_estimators=500,
    max_features=4,
    min_samples_leaf=5,
    max_depth=9,
    verbose=1, 
    n_jobs=-1, 
    random_state=RANDOM_SEED)

# обучаем модель на тестовом наборе данных
model_rf_Tg.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_rf_Tg.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 426 tasks      | elapsed:    0.3s
[Parallel(n_jobs=-1)]: Done 500 out of 500 | elapsed:    0.4s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 426 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 500 out of 500 | elapsed:    0.0s finished


In [53]:
print('model_rf_Tg MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('model_rf_Tg MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

model_rf_Tg MSE: 0.028
model_rf_Tg MAPE: 0.013


In [54]:
# save model
with open('autoclave_model_rf_Tg.pkl','wb') as f:
    pickle.dump(model_rf_Tg,f)

In [55]:
model_lr_Tg = LinearRegression()

# обучаем модель на тестовом наборе данных
model_lr_Tg.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_lr_Tg.predict(X_test)

print('model_lr_Tg MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('model_lr_Tg MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

model_lr_Tg MSE: 4.113
model_lr_Tg MAPE: 0.997


In [56]:
# save model
with open('autoclave_model_lr_Tg.pkl','wb') as f:
    pickle.dump(model_lr_Tg,f)

In [57]:
print('Линейная регрессия. Температура стеклования:', model_lr_Tg.predict(x2))
print('Случайный лес регрессия. Температура стеклования:', model_rf_Tg.predict(x2))

Линейная регрессия. Температура стеклования: [162.54593186]
Случайный лес регрессия. Температура стеклования: [160.45217411]


[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 426 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 500 out of 500 | elapsed:    0.0s finished
